# บทที่ 9 · ลงมือทำ: pipeline เล็ก ๆ ตั้งแต่ต้นจนจบ

รวมทุกบทเข้าด้วยกัน: **ไฟล์ CSV + API → bronze → silver → gold** บน RustFS + Iceberg

```
 marketplace ส่งไฟล์ CSV ──> s3://landing/marketplace/ ──┐
                                                       ├──> bronze ──> silver ──> gold
 API ค่าโฆษณา ─────────────────────────────────────────┘
```

| ขั้น | ทำอะไร | แบบไหน |
|---|---|---|
| bronze | เก็บของดิบ + บันทึกว่ามาจากไหน เมื่อไหร่ | เพิ่มต่อท้ายอย่างเดียว |
| silver | แปลงชนิด ตัดซ้ำ ทำเครื่องหมายแถวเสีย | **incremental**: ทำเฉพาะของใหม่ + MERGE |
| gold | สรุปยอดขายเทียบค่าโฆษณารายวัน | **full load**: ลบทั้งหมดแล้วสร้างใหม่ |

กติกาสำคัญ: **รันซ้ำกี่รอบ ผลต้องเหมือนเดิม** (ไม่มีข้อมูลซ้ำ ไม่หาย) และทุกขั้นเป็น **function** ที่เรียกใช้ซ้ำได้

In [ ]:
import io
import logging
from datetime import datetime
from pathlib import Path

import pandas as pd

from lakehouse import connect_lakehouse, make_bucket, make_table_bucket, show_tables, storage

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s: %(message)s", force=True)
log = logging.getLogger("pipeline")

LANDING = "landing"                       # bucket รับไฟล์
MARKETPLACE_FOLDER = "marketplace/"       # ไฟล์ใหม่มาวางที่นี่
REQUIRED_COLUMNS = ["marketplace", "order_ref", "order_datetime", "sku", "qty", "unit_price", "platform_fee", "status"]

## 9.1 เตรียมที่เก็บ (รันซ้ำได้)

In [ ]:
def prepare_storage() -> None:
    make_bucket(LANDING)
    make_table_bucket("lakehouse")
    lake = connect_lakehouse()
    for layer in ["bronze", "silver", "gold"]:
        lake.sql(f"CREATE SCHEMA IF NOT EXISTS lake.{layer}")
    lake.close()
    log.info("storage ready")


prepare_storage()

## 9.2 จำลอง "marketplace ส่งไฟล์มา"

ในชีวิตจริงคนอื่นเป็นคนวางไฟล์ลง bucket เราจำลองด้วยการอัปโหลดไฟล์จากโฟลเดอร์ `data/`

In [ ]:
def partner_sends_file(file_name: str) -> str:
    """อัปโหลดไฟล์จาก data/marketplace/ ไปวางที่ s3://landing/marketplace/"""
    key = MARKETPLACE_FOLDER + file_name
    storage().upload_file(f"../data/marketplace/{file_name}", LANDING, key)
    log.info("partner dropped %s", key)
    return key


for day in ["2026-09-23", "2026-09-24", "2026-09-25"]:
    partner_sends_file(f"marketplace_orders_{day}.csv")

## 9.3 ไฟล์ → bronze

1. ดูว่ามีไฟล์รออยู่ไหม
2. อ่านไฟล์ (เก็บทุกคอลัมน์เป็นข้อความตามต้นฉบับ) ตรวจคอลัมน์
3. เพิ่มคอลัมน์ `source_file`, `loaded_at` แล้วเพิ่มลง `bronze.marketplace_orders`
4. ย้ายไฟล์ไป `done/` (สำเร็จ) หรือ `rejected/` (ไฟล์เสีย) จะได้ไม่อ่านซ้ำ

In [ ]:
def files_waiting() -> list[str]:
    found = storage().list_objects_v2(Bucket=LANDING, Prefix=MARKETPLACE_FOLDER).get("Contents", [])
    return sorted(f["Key"] for f in found if f["Key"].endswith(".csv"))


def move_file(key: str, to_folder: str) -> None:
    s3 = storage()
    s3.copy_object(Bucket=LANDING, Key=f"{to_folder}/{key}", CopySource={"Bucket": LANDING, "Key": key})
    s3.delete_object(Bucket=LANDING, Key=key)


def table_exists(lake, table: str) -> bool:
    layer, name = table.split(".")
    found = lake.sql(f"""SELECT count(*) FROM information_schema.tables
                         WHERE table_catalog = 'lake' AND table_schema = '{layer}' AND table_name = '{name}'""")
    return found.fetchone()[0] > 0


def add_to_bronze(lake, new_rows: pd.DataFrame, table: str) -> int:
    """เพิ่มแถวต่อท้ายตาราง bronze (ครั้งแรกสร้างตารางให้ด้วย)"""
    if not table_exists(lake, f"bronze.{table}"):
        lake.sql(f"CREATE TABLE lake.bronze.{table} AS SELECT * FROM new_rows LIMIT 0")
    lake.sql(f"INSERT INTO lake.bronze.{table} BY NAME SELECT * FROM new_rows")
    return len(new_rows)


print(files_waiting())

In [ ]:
def load_files_to_bronze() -> dict:
    """ย้ายทุกไฟล์ที่รออยู่เข้า bronze คืนสรุปว่าไฟล์ไหนเป็นอย่างไร"""
    summary = {"loaded": [], "rejected": [], "rows": 0}
    lake = connect_lakehouse()
    for key in files_waiting():
        file_bytes = storage().get_object(Bucket=LANDING, Key=key)["Body"].read()
        rows = pd.read_csv(io.BytesIO(file_bytes), dtype=str)          # dtype=str: เก็บตามต้นฉบับทุกตัวอักษร
        missing = [c for c in REQUIRED_COLUMNS if c not in rows.columns]
        if missing:
            log.warning("%s missing columns %s -> rejected", key, missing)
            move_file(key, "rejected")
            summary["rejected"].append(key)
            continue
        rows = rows[REQUIRED_COLUMNS].assign(source_file=key, loaded_at=datetime.now())
        summary["rows"] += add_to_bronze(lake, rows, "marketplace_orders")
        move_file(key, "done")
        summary["loaded"].append(key)
        log.info("%s -> bronze (%d rows)", key, len(rows))
    lake.close()
    return summary


load_files_to_bronze()

In [ ]:
print("ไฟล์ที่ยังรออยู่:", files_waiting())                 # ว่างแล้ว ย้ายไป done/ หมด
load_files_to_bronze()                                      # รันซ้ำ: ไม่มีไฟล์ใหม่ = ไม่มีอะไรเกิดขึ้น

## 9.4 API → bronze

In [ ]:
import os
import time

import requests
from dotenv import load_dotenv

load_dotenv()
API_URL = os.getenv("SHOP_API_URL", "http://127.0.0.1:8000")
API_HEADERS = {"X-API-Key": os.getenv("SHOP_API_KEY", "class-demo-key")}


def get_json(path: str, params: dict, max_tries: int = 6) -> dict:
    for attempt in range(1, max_tries + 1):
        resp = requests.get(f"{API_URL}{path}", params=params, headers=API_HEADERS, timeout=10)
        if resp.status_code in (429, 500, 502, 503, 504) and attempt < max_tries:
            time.sleep(int(resp.headers.get("Retry-After", 1)))
            continue
        resp.raise_for_status()
        return resp.json()


def load_ads_to_bronze(day: str) -> int:
    rows, page = [], 1
    while True:
        body = get_json("/v1/ad-spend", {"date": day, "page": page})
        rows.extend(body["data"])
        if page >= body["total_pages"]:
            break
        page += 1
    new_rows = pd.DataFrame(rows).assign(loaded_at=datetime.now())
    lake = connect_lakehouse()
    count = add_to_bronze(lake, new_rows, "ad_spend")
    lake.close()
    log.info("ads %s -> bronze (%d rows)", day, count)
    return count


for day in ["2026-09-23", "2026-09-24", "2026-09-25"]:
    load_ads_to_bronze(day)

In [ ]:
lake = connect_lakehouse()
show_tables(lake)

In [ ]:
lake.sql("SELECT source_file, count(*) AS rows, min(loaded_at) AS loaded_at FROM lake.bronze.marketplace_orders GROUP BY 1 ORDER BY 1")

## 9.5 bronze → silver (incremental + MERGE)

- **incremental**: เอาเฉพาะแถวที่เข้า bronze **หลัง** ครั้งล่าสุดที่ทำ silver (ดูจาก `loaded_at` ที่มากที่สุดใน silver)
- แปลงข้อความเป็นชนิดที่ถูก (`TRY_CAST` = แปลงไม่ได้ให้เป็น NULL แทนที่จะพังทั้งไฟล์)
- แถวซ้ำเอาแถวล่าสุด (`row_number()` จาก session 3)
- แถวเสีย **ไม่ทิ้ง** แต่เขียนเหตุผลไว้ในคอลัมน์ `problem`
- `MERGE` เข้า silver: รันซ้ำไม่เกิดแถวซ้ำ

In [ ]:
SILVER_ORDERS_SQL = """
CREATE TABLE IF NOT EXISTS lake.silver.marketplace_orders (
    marketplace VARCHAR, order_ref VARCHAR, sku VARCHAR, order_datetime TIMESTAMP,
    qty INTEGER, unit_price DECIMAL(12,2), line_total DECIMAL(14,2), status VARCHAR,
    problem VARCHAR, source_file VARCHAR, loaded_at TIMESTAMP
);

MERGE INTO lake.silver.marketplace_orders AS old
USING (
    SELECT
        marketplace, order_ref, sku,
        TRY_CAST(order_datetime AS TIMESTAMP)               AS order_datetime,
        TRY_CAST(qty AS INTEGER)                            AS qty,
        TRY_CAST(unit_price AS DECIMAL(12,2))               AS unit_price,
        TRY_CAST(qty AS INTEGER) * TRY_CAST(unit_price AS DECIMAL(12,2)) AS line_total,
        status,
        CASE WHEN TRY_CAST(qty AS INTEGER) IS NULL OR TRY_CAST(qty AS INTEGER) <= 0 THEN 'bad qty'
             WHEN TRY_CAST(unit_price AS DECIMAL(12,2)) IS NULL THEN 'bad price'
        END                                                 AS problem,
        source_file, loaded_at
    FROM lake.bronze.marketplace_orders
    WHERE loaded_at > (SELECT coalesce(max(loaded_at), TIMESTAMP '1900-01-01') FROM lake.silver.marketplace_orders)
    QUALIFY row_number() OVER (PARTITION BY marketplace, order_ref, sku ORDER BY loaded_at DESC) = 1
) AS new
ON old.marketplace = new.marketplace AND old.order_ref = new.order_ref AND old.sku = new.sku
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;
"""

SILVER_ADS_SQL = """
CREATE TABLE IF NOT EXISTS lake.silver.ad_spend (
    spend_date DATE, campaign_id VARCHAR, channel VARCHAR, spend DECIMAL(12,2), loaded_at TIMESTAMP
);

MERGE INTO lake.silver.ad_spend AS old
USING (
    SELECT CAST("date" AS DATE) AS spend_date, campaign_id, channel,
           CAST(spend AS DECIMAL(12,2)) AS spend, loaded_at
    FROM lake.bronze.ad_spend
    WHERE loaded_at > (SELECT coalesce(max(loaded_at), TIMESTAMP '1900-01-01') FROM lake.silver.ad_spend)
    QUALIFY row_number() OVER (PARTITION BY "date", campaign_id ORDER BY loaded_at DESC) = 1
) AS new
ON old.spend_date = new.spend_date AND old.campaign_id = new.campaign_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;
"""


def update_silver() -> dict:
    lake = connect_lakehouse()
    lake.sql(SILVER_ORDERS_SQL)
    lake.sql(SILVER_ADS_SQL)
    counts = {
        "orders": lake.sql("SELECT count(*) FROM lake.silver.marketplace_orders").fetchone()[0],
        "ad_spend": lake.sql("SELECT count(*) FROM lake.silver.ad_spend").fetchone()[0],
    }
    lake.close()
    log.info("silver: %s", counts)
    return counts


update_silver()

## 9.6 silver → gold (full load)

ตาราง gold เล็ก สร้างใหม่ทั้งหมดทุกครั้งง่ายและถูกเสมอ: `DELETE` ทั้งตาราง แล้ว `INSERT` ใหม่ **ในรายการเดียว (transaction)**
คนที่เปิดดูระหว่างนั้นจะไม่เห็นตารางว่าง

In [ ]:
GOLD_SQL = """
CREATE TABLE IF NOT EXISTS lake.gold.daily_summary (
    sales_date DATE, orders BIGINT, revenue DECIMAL(14,2), ad_spend DECIMAL(14,2),
    revenue_per_ad_baht DECIMAL(10,2), built_at TIMESTAMP
);

BEGIN TRANSACTION;
DELETE FROM lake.gold.daily_summary;
INSERT INTO lake.gold.daily_summary
WITH sales AS (
    SELECT CAST(order_datetime AS DATE) AS sales_date,
           count(DISTINCT marketplace || order_ref) AS orders,
           sum(line_total) AS revenue
    FROM lake.silver.marketplace_orders
    WHERE status <> 'cancelled' AND problem IS NULL
    GROUP BY 1
),
ads AS (
    SELECT spend_date, sum(spend) AS ad_spend FROM lake.silver.ad_spend GROUP BY 1
)
SELECT s.sales_date, s.orders, s.revenue, a.ad_spend,
       s.revenue / nullif(a.ad_spend, 0) AS revenue_per_ad_baht,
       now()::TIMESTAMP AS built_at
FROM sales AS s
LEFT JOIN ads AS a ON a.spend_date = s.sales_date;
COMMIT;
"""


def build_gold() -> int:
    lake = connect_lakehouse()
    lake.sql(GOLD_SQL)
    rows = lake.sql("SELECT count(*) FROM lake.gold.daily_summary").fetchone()[0]
    lake.close()
    log.info("gold: %d days", rows)
    return rows


build_gold()

In [ ]:
lake = connect_lakehouse()
lake.sql("SELECT * FROM lake.gold.daily_summary ORDER BY sales_date")

`revenue_per_ad_baht` = จ่ายค่าโฆษณา 1 บาท ขายบน marketplace ได้กี่บาท

## 9.7 รวมทุกขั้นเป็น function เดียว แล้วลองรันซ้ำ

In [ ]:
def run_everything(day: str) -> dict:
    """งานประจำวัน: ไฟล์ -> bronze, API ของวันนั้น -> bronze, silver, gold"""
    return {
        "day": day,
        "files": load_files_to_bronze(),
        "ad_rows": load_ads_to_bronze(day),
        "silver": update_silver(),
        "gold_days": build_gold(),
    }


def count_rows() -> dict:
    lake = connect_lakehouse()
    tables = ["bronze.marketplace_orders", "bronze.ad_spend", "silver.marketplace_orders", "silver.ad_spend",
              "gold.daily_summary"]
    counts = {t: lake.sql(f"SELECT count(*) FROM lake.{t}").fetchone()[0] for t in tables}
    lake.close()
    return counts


before = count_rows()
run_everything("2026-09-25")          # รันวันเดิมซ้ำ (เช่น ต้องรันใหม่เพราะเมื่อกี้เน็ตหลุด)
after = count_rows()
pd.DataFrame({"before": before, "after": after})

สังเกต:
- **bronze.ad_spend เพิ่ม** 12 แถว (ดึง API วันเดิมซ้ำ bronze เก็บทุกครั้งที่ดึง ตามหน้าที่ของ bronze)
- **silver / gold ไม่เพิ่ม** เพราะ MERGE เจอว่ามีอยู่แล้ว และ gold สร้างใหม่ทั้งหมด → ตัวเลขไม่ซ้ำ ✅

## 9.8 วันใหม่มาถึง: ไฟล์ดี 1 ไฟล์ + ไฟล์เสีย 1 ไฟล์

In [ ]:
partner_sends_file("marketplace_orders_2026-09-26.csv")               # มีแถวเสียข้างใน (qty = 0)
partner_sends_file("marketplace_orders_2026-09-30_v2_broken.csv")     # ชื่อคอลัมน์ผิดทั้งไฟล์
run_everything("2026-09-26")

In [ ]:
lake = connect_lakehouse()
lake.sql("SELECT * FROM lake.gold.daily_summary ORDER BY sales_date")

In [ ]:
lake.sql("SELECT marketplace, order_ref, sku, qty, problem, source_file FROM lake.silver.marketplace_orders WHERE problem IS NOT NULL")

In [ ]:
for f in storage().list_objects_v2(Bucket=LANDING).get("Contents", []):
    print(f["Key"])

- แถวเสียอยู่ใน silver พร้อม `problem` (ตรวจสอบได้) แต่ gold ไม่นับ
- ไฟล์ที่คอลัมน์ผิดถูกย้ายไป `rejected/` ไม่เข้า bronze เลย → แจ้งคนส่งไฟล์ให้แก้แล้วส่งใหม่

## 9.9 เริ่มใหม่ทั้งหมด (ถ้าอยากลองอีกรอบ)

เปลี่ยน `False` เป็น `True` แล้วรัน จะลบตารางทั้งหมดและไฟล์ใน landing แล้วกลับไปรันตั้งแต่ 9.1 ได้

In [ ]:
START_OVER = False

if START_OVER:
    lake = connect_lakehouse()
    for layer, table in lake.sql("""SELECT table_schema, table_name FROM information_schema.tables
                                    WHERE table_catalog = 'lake'""").fetchall():
        lake.sql(f"DROP TABLE lake.{layer}.{table}")
    for f in storage().list_objects_v2(Bucket=LANDING).get("Contents", []):
        storage().delete_object(Bucket=LANDING, Key=f["Key"])
    print("ลบแล้ว")

## ✅ สรุปบทที่ 9

| ขั้น | function | หลักการ |
|---|---|---|
| รับไฟล์ | `files_waiting()`, `load_files_to_bronze()`, `move_file()` | ตรวจคอลัมน์, ย้ายไป done/rejected จะได้ไม่อ่านซ้ำ |
| API | `load_ads_to_bronze(day)` | หลายหน้า + ลองใหม่เมื่อเว็บไม่ว่าง |
| bronze | `add_to_bronze()` | เพิ่มอย่างเดียว + `source_file`, `loaded_at` |
| silver | `update_silver()` | incremental (`loaded_at`) + ตัดซ้ำ + MERGE + ทำเครื่องหมายแถวเสีย |
| gold | `build_gold()` | full load: DELETE + INSERT ใน transaction |
| ทั้งหมด | `run_everything(day)` | รันซ้ำได้ ผลเหมือนเดิม |

🧪 **แบบฝึกท้ายบท:** ส่งไฟล์วันที่ `2026-09-27` และ `2026-09-28` เข้ามา แล้วรัน `run_everything` สำหรับทั้งสองวัน
ตรวจว่า gold มีกี่วัน และวันไหน `revenue_per_ad_baht` สูงที่สุด

In [ ]:
# เขียนต่อตรงนี้